In [1]:
!pip install -q boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os

os.environ["AWS_ACCESS_KEY_ID"] = "YOUR ACCESS KEY ID"
os.environ["AWS_SECRET_ACCESS_KEY"] = "YOUR SECRET ACCESS KEY"
os.environ["AWS_DEFAULT_REGION"] = "REGION THAT YOU HAVE SET"

In [3]:
import boto3
import json
import pandas as pd

In [4]:
bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

In [5]:
MODEL_ID = "us.amazon.nova-2-lite-v1:0"  # Cross-region inference profile

In [15]:
def invoke_bedrock(system_prompt, user_prompt):

    # Prepend system prompt to user prompt for models that don't support
    # a separate system role or expect direct instruction within the prompt.
    full_user_prompt = f"{system_prompt}\n\n{user_prompt}"

    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "text": full_user_prompt
                    }
                ]
            }
        ],
        "inferenceConfig": {
            "max_new_tokens": 300,
            "temperature": 0.3
        }
    }

    response = bedrock.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body)
    )

    response_body = json.loads(response["body"].read())

    return response_body["output"]["message"]["content"][0]["text"]


In [9]:
review = """
Project hail mary is a great film but I dont understand how such a mediocre film
could be compared with the greatness of Interstellar
"""


In [10]:
zero_prompt = f"""
Classify this review as:
Positive, Negative, or Neutral.

Review:
{review}

Give:
1. Category
2. Short explanation
"""


In [11]:
few_prompt = f"""
Example 1:
Review: "Interstellar is the best sci-fi movie."
Category: Positive

Example 2:
Review: "Rebel moon is one of worst sci-fi film."
Category: Negative

Example 3:
Review: "Passengers is an okay sci-fi film."
Category: Neutral

Now classify:

Review:
{review}

Return:
1. Category
2. Explanation
"""

In [12]:
cot_prompt = f"""
Analyze the review step-by-step.

1. Find positive points
2. Find negative points
3. Decide overall sentiment

Review:
{review}

Return:
1. Reasoning
2. Final category
"""

In [13]:
analyst_system = """
You are a senior data analyst.
Be concise and objective.
"""

writer_system = """
You are a creative writer.
Be descriptive and expressive.
"""

In [16]:
# ZERO SHOT
zero_analyst = invoke_bedrock(
    analyst_system,
    zero_prompt
)

zero_writer = invoke_bedrock(
    writer_system,
    zero_prompt
)

# FEW SHOT
few_analyst = invoke_bedrock(
    analyst_system,
    few_prompt
)

few_writer = invoke_bedrock(
    writer_system,
    few_prompt
)

# CHAIN OF THOUGHT
cot_analyst = invoke_bedrock(
    analyst_system,
    cot_prompt
)

cot_writer = invoke_bedrock(
    writer_system,
    cot_prompt
)

In [17]:
results = pd.DataFrame({
    "Technique": [
        "Zero-Shot",
        "Zero-Shot",
        "Few-Shot",
        "Few-Shot",
        "Chain-of-Thought",
        "Chain-of-Thought"
    ],

    "Persona": [
        "Senior Data Analyst",
        "Creative Writer",
        "Senior Data Analyst",
        "Creative Writer",
        "Senior Data Analyst",
        "Creative Writer"
    ],

    "Response": [
        zero_analyst,
        zero_writer,
        few_analyst,
        few_writer,
        cot_analyst,
        cot_writer
    ]
})

results

,Technique,Persona,Response
0,Zero-Shot,Senior Data Analyst,### **Category: Negative**\n\n### **Short Expl...
1,Zero-Shot,Creative Writer,### **Category: Negative**\n\n---\n\n### **Sho...
2,Few-Shot,Senior Data Analyst,### **1. Category: Mixed**\n\n### **2. Explana...
3,Few-Shot,Creative Writer,### **1. Category: Mixed (leaning Positive)** ...
4,Chain-of-Thought,Senior Data Analyst,### **Analysis of the Review**\n\n---\n\n#### ...
5,Chain-of-Thought,Creative Writer,"### **Analysis of the Review: ""Project Hail Ma..."


In [18]:
print("===== ZERO SHOT : ANALYST =====")
print(zero_analyst)

print("\n===== ZERO SHOT : WRITER =====")
print(zero_writer)

print("\n===== FEW SHOT : ANALYST =====")
print(few_analyst)

print("\n===== FEW SHOT : WRITER =====")
print(few_writer)

print("\n===== COT : ANALYST =====")
print(cot_analyst)

print("\n===== COT : WRITER =====")
print(cot_writer)

===== ZERO SHOT : ANALYST =====
### **Category: Negative**

### **Short Explanation:**  
While the reviewer acknowledges *"Project Hail Mary"* as a *"great film,"* the overall sentiment is **negative** because the core message expresses disappointment and criticism. The reviewer conveys frustration by stating they don’t understand how a *"mediocre film"* (Project Hail Mary) could be compared to the *"greatness of Interstellar,"* implying that the comparison is unjustified and devalues the superior quality of *Interstellar*. The negative tone dominates due to the contrast and implied criticism.

===== ZERO SHOT : WRITER =====
### **Category: Negative**

---

### **Short Explanation:**

The review is classified as **Negative** because, despite opening with a seemingly positive statement ("Project Hail Mary is a great film"), the core sentiment expresses disappointment and criticism. The reviewer conveys frustration by highlighting a perceived injustice in comparing *Project Hail Mary* to

From the looks of it we can see that a writer sees the content from a creative perspective with a verbose language and tonality while an analyst has a very critical and to the point overview of the review. Making it that LLM's do change the response when changing the role the LLM is assigned.